# Reto Analytics Corona - Limpieza Exploratoria

Este notebook carga y limpia las dos bases suministradas por Corona:
- `base_transaccional.csv` (~2.6M filas)
- `maestro_clientes.csv` (~552K filas)

**Flujo de trabajo:** este notebook es el espacio de exploración interactiva con [Data Wrangler](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.datawrangler). Una vez validadas las transformaciones aqui, se consolidan en `scripts/limpieza_datos.py` para correr el pipeline completo de forma reproducible.

**Nota:** los archivos CSV deben estar en la carpeta `data/` (no se suben a git por su tamaño - ver README para el enlace de descarga).

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Cargar los datos

Los archivos vienen separados por `;` (no por coma), y los decimales usan `.` (no `,`).

Tip: para explorar visualmente cualquiera de los dos DataFrames, corre la celda que lo crea y luego, en el panel de Variables de VS Code (o clic derecho sobre la variable), selecciona **"Open in Data Wrangler"**.

In [ ]:
transaccional = pd.read_csv(
    "../data/base_transaccional.csv",
    sep=";",
    encoding="utf-8-sig",
    decimal="."
)

transaccional.shape

In [ ]:
clientes = pd.read_csv(
    "../data/maestro_clientes.csv",
    sep=";",
    encoding="utf-8-sig",
    decimal="."
)

clientes.shape

## 3. Vista rápida de las columnas y tipos

In [ ]:
transaccional.head()

In [ ]:
transaccional.dtypes

In [ ]:
clientes.head()

In [ ]:
clientes.dtypes

## 4. Corregir nombres de columnas

El archivo original trae la columna `Subategoria` con un error de tipeo (falta la "c"). La renombramos aquí para que el resto del análisis sea consistente.

In [ ]:
transaccional = transaccional.rename(columns={"Subategoria": "Subcategoria"})

# Estandarizar nombres de columnas: minúsculas y sin espacios
transaccional.columns = (
    transaccional.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

clientes.columns = (
    clientes.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print(transaccional.columns.tolist())
print(clientes.columns.tolist())

## 5. Limpieza de valores y tipos de dato

In [ ]:
# "(No Value)" en Region se trata como dato faltante
transaccional["region"] = transaccional["region"].replace("(No Value)", np.nan)

# Convertir fechas (formato AAAA-MM-DD)
transaccional["fecha_pedido"] = pd.to_datetime(transaccional["fecha_pedido"], format="%Y-%m-%d", errors="coerce")
clientes["primera_fecha"] = pd.to_datetime(clientes["primera_fecha"], format="%Y-%m-%d", errors="coerce")

# Asegurar que las columnas numéricas queden como float
for col in ["cantidad", "precio", "valor"]:
    transaccional[col] = pd.to_numeric(transaccional[col], errors="coerce")

clientes["edad"] = pd.to_numeric(clientes["edad"], errors="coerce")

## 6. Revisión de calidad de datos

In [ ]:
# Valores faltantes por columna
print("--- Transaccional ---")
print(transaccional.isna().sum())
print("\n--- Clientes ---")
print(clientes.isna().sum())

In [ ]:
# Duplicados
print("Duplicados en transaccional:", transaccional.duplicated().sum())
print("Duplicados en clientes:", clientes.duplicated().sum())

In [ ]:
# Clientes que aparecen en transaccional pero no en el maestro (y viceversa)
clientes_transaccional = set(transaccional["cliente"].unique())
clientes_maestro = set(clientes["cliente"].unique())

print("En transaccional pero no en maestro:", len(clientes_transaccional - clientes_maestro))
print("En maestro pero no en transaccional:", len(clientes_maestro - clientes_transaccional))

## 7. Estadísticas descriptivas rápidas

In [ ]:
transaccional[["cantidad", "precio", "valor"]].describe()

In [ ]:
transaccional["negocio"].value_counts()

In [ ]:
clientes["tipo_cliente"].value_counts()

In [ ]:
clientes["edad"].describe()

## 8. Unir las dos bases (cliente-transacción)

In [ ]:
datos_completos = transaccional.merge(clientes, on="cliente", how="left")
datos_completos.shape

In [ ]:
datos_completos.head()

## 9. Guardar el resultado limpio

Se guarda en formato `.parquet` (más rápido y compacto que CSV para 2.6M+ filas) dentro de `data/` (ignorado por git).

In [ ]:
datos_completos.to_parquet("../data/datos_completos.parquet", index=False)
print("Guardado en data/datos_completos.parquet")

---
### Próximos pasos
- Ingeniería de características (RFM: recencia, frecuencia, valor monetario)
- Segmentación (K-means / clustering jerárquico)
- Consolidar transformaciones finales en `scripts/limpieza_datos.py`